# Lab7.1 - Clasificarea imaginilor din fisiere
## CNV-W2A2 / CIFAR-10 / PYNQ-Z2

**Obiectiv:** utilizarea modelului preantrenat CNV-W2A2 din `onigas/BNN-PYNQ`, cu ponderi si activari de 2 biti. Nu antrenam o retea noua si nu regeneram bitstream-ul.

**Platforma:** PYNQ-Z2, imaginea PYNQ 2.7.0 a cursului, Jupyter/Python. Notebook-ul este independent de Lab1-Lab6 si nu modifica fisierele acestora.

Parcurgem un test de referinta, clasificam JPG/PNG proprii, evaluam un director etichetat si salvam CSV-uri cu predictii si timpi. Camera si video vor fi tratate separat.

> Este clasificare intre zece clase, nu detectie de pozitii sau de mai multe obiecte. O imagine necunoscuta primeste tot o clasa cunoscuta. Scorurile NU sunt probabilitati calibrate.

Sursa fixa: [BNN-PYNQ, commit d39c1c7](https://github.com/onigas/BNN-PYNQ/tree/d39c1c75403db571f54dd0f0cf27ae5104ac968e).

## 1. Pregatirea mediului

Copiati intregul director `Lab7` pe placa si deschideti acest notebook in Jupyter. `lab71.py` trebuie sa fie langa notebook. Executati celulele in ordine.

Opriti celelalte notebook-uri care programeaza FPGA-ul: clasificatorul inlocuieste overlay-ul activ. Nu utilizati simultan overlay-ul Resize/Sobel.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from lab71 import (ROOT, CLASS_NAMES, EXTENSIONS, prepare_reference, load_bnn,
                   check_overlay, quiet_call, read_rgb, network_input, infer_file,
                   load_labels, expected_label, save_csv, download_original_jpg)

print("Python:", sys.executable)
print("BOARD:", os.environ.get("BOARD", "nesetat"))
if os.environ.get("BOARD") != "Pynq-Z2":
    raise RuntimeError("Rulati pe PYNQ-Z2 in kernelul placii. Nu simulati placa setand BOARD pe PC.")
reference_png = prepare_reference()
print("Referinta locala verificata:", reference_png)
for class_id, name in enumerate(CLASS_NAMES):
    print("{:2d}  {}".format(class_id, name))

## 2. Accesul la modelul existent

Folosim mai intai pachetul `bnn` deja instalat. Daca lipseste, celula descarca arhiva la commit-ul fixat si extrage numai componentele CNV-W2A2 deja construite in `Lab7/_vendor/BNN-PYNQ`.

Nu ruleaza `pip`, `setup.py`, Vivado, FINN sau antrenare si nu actualizeaza pachetele placii. Prima descarcare necesita Internet; apoi se foloseste copia locala. Pentru utilizare offline consultati `README.md`.

Originea runtime-ului, versiunile si hash-urile se salveaza in `results/environment.json`. O instalare preexistenta nu este declarata automat identica cu acel commit.

In [ ]:
ALLOW_DOWNLOAD = True
bnn, provenance = load_bnn(allow_download=ALLOW_DOWNLOAD)
print("BNN:", bnn.__file__)
print("Origine:", provenance["origin"])
print("PYNQ:", provenance["pynq_version"])
print("Model: CNV-W2A2 / CIFAR-10 / hardware")

## 3. Incarcarea clasificatorului in FPGA

Constructorul original incarca overlay-ul si ponderile. Il apelam o singura data, nu pentru fiecare imagine. Initializarea nu intra in benchmark.

La eroare de biblioteca sau overlay opriti experimentul si pastrati mesajul complet. Nu trecem automat la software si nu raportam ARM drept FPGA.

In [ ]:
if "classifier" not in globals():
    classifier = bnn.CnvClassifier(bnn.NETWORK_CNVW2A2, "cifar10", bnn.RUNTIME_HW)
if (classifier.net, classifier.params, classifier.runtime) != (bnn.NETWORK_CNVW2A2, "cifar10", bnn.RUNTIME_HW):
    raise RuntimeError("Alt clasificator exista in kernel. Restartati kernelul.")
check_overlay(classifier)
if list(classifier.classes) != CLASS_NAMES:
    raise RuntimeError("Ordinea claselor nu corespunde CIFAR-10.")
print("Overlay:", classifier.bnn.bitstream_path)

## 4. Referinta: cerbul din proiectul original

`deer.bin` este fisierul original din `tests/Test_image/`, verificat prin hash. `deer.png` este derivat fara pierderi din exact aceiasi pixeli RGB 32 x 32, nu este o imagine noua.

Fisierul binar are 3073 octeti: un octet de antet si trei plane R, G, B. Antetul exemplului are valoarea 1, dar **nu il folosim drept eticheta reala**. Cerbul corespunde clasei 4 / Deer; testul original pentru fotografia `deer.jpg` verifica aceasta clasa.

Verificam scorurile BIN vs PNG si clasa obtinuta. Acest test de functionare nu estimeaza acuratetea pe CIFAR-10 complet.

In [ ]:
check_overlay(classifier)
reference_bin = ROOT / "images/reference/deer.bin"
scores_bin = np.asarray(quiet_call(classifier.classify_cifar_details, str(reference_bin))).copy()
reference_report, reference_image, reference_scores = infer_file(classifier, reference_png)
same_scores = np.array_equal(scores_bin, reference_scores)
print("Scoruri BIN == PNG:", same_scores)
print("Asteptat: 4 / Deer | Obtinut:", reference_report["prediction_id"], reference_report["prediction"])
if not same_scores:
    raise RuntimeError("Scoruri diferite BIN/PNG. Verificati preprocesarea si runtime-ul.")
if reference_report["prediction_id"] != 4:
    raise RuntimeError("Testul cerbului nu a trecut. Verificati modelul inainte de continuare.")
print("TEST DE REFERINTA: PASS")
save_csv(ROOT / "results/reference.csv", [dict(reference_report, expected_id=4, same_scores=same_scores)])

def show_result(report, image, scores):
    plt.figure(figsize=(5, 4))
    plt.imshow(image)
    plt.title("{} -> {}".format(Path(report["file"]).name, report["prediction"]))
    plt.axis("off")
    plt.show()
    plt.figure(figsize=(4, 4))
    plt.imshow(network_input(classifier, image), interpolation="nearest")
    plt.title("Intrarea efectiva: 32 x 32 RGB")
    plt.axis("off")
    plt.show()
    indices = np.argsort(scores)[-5:]
    plt.figure(figsize=(7, 3))
    plt.barh([classifier.classes[int(i)] for i in indices], scores[indices])
    plt.xlabel("Scor brut - NU procent de incredere")
    plt.title("Primele cinci clase")
    plt.tight_layout()
    plt.show()
    print("Citire: {:.3f} ms | API: {:.3f} ms | fisier -> predictie: {:.3f} ms".format(
        report["decode_ms"], report["api_ms"], report["file_to_prediction_ms"]))

show_result(reference_report, reference_image, reference_scores)

### Optional: fotografia JPG exacta din testul original

Activati cu acces la Internet sau cu fotografia originala copiata in `images/reference/deer_original.jpg`. Helper-ul verifica hash-ul JPG. Reproduceti astfel testul pentru `deer.jpg` din `tests/test_qnn.py`.

In [ ]:
TEST_ORIGINAL_JPG = False
if TEST_ORIGINAL_JPG:
    jpg_path = download_original_jpg()
    jpg_report, jpg_image, jpg_scores = infer_file(classifier, jpg_path)
    show_result(jpg_report, jpg_image, jpg_scores)
    if jpg_report["prediction_id"] != 4:
        raise RuntimeError("Testul original deer.jpg nu a returnat Deer.")
    save_csv(ROOT / "results/reference_jpg.csv", [dict(jpg_report, expected_id=4)])

## 5. O imagine proprie din fisier

Incarcati JPG/PNG in `images/own/` si schimbati `IMAGE_PATH`. Initial este aleasa referinta, pentru rulare fara fisiere personale.

Alegeti un obiect din clasele suportate. Persoana, telefonul, scaunul si fructele NU sunt clase CIFAR-10. Preprocesarea originala realizeaza RGB, redimensionare cu pastrarea proportiilor si incadrare in 32 x 32. Nu adaugam Sobel, grayscale sau normalizare externa. Pillow citeste RGB; la imagini citite cu OpenCV ar trebui convertit BGR -> RGB.

In [ ]:
IMAGE_PATH = reference_png
# Dupa incarcarea unei fotografii proprii:
# IMAGE_PATH = ROOT / "images/own/my_cat.jpg"
report, image, scores = infer_file(classifier, IMAGE_PATH)
show_result(report, image, scores)
save_csv(ROOT / "results/single_image.csv", [report])

## 6. Clasificarea unui director si evaluarea

Adaugati imagini in subdirectoare precum `images/own/Cat/` si `images/own/Truck/`, sau completati `images/labels.csv`: cale relativa la `images/` si `class_id` intre 0 si 9. CSV-ul are prioritate fata de numele subdirectorului.

Fisierele fara eticheta sunt clasificate, dar nu intra in acuratete. Fisierele corupte sunt raportate separat. Erorile de FPGA opresc experimentul.

Acuratetea este raportata numai pentru acest lot: corecte / imagini etichetate procesate. Un singur cerb nu este un benchmark reprezentativ. JPG si PNG pentru acelasi cerb nu sunt exemple independente.

In [ ]:
from PIL import UnidentifiedImageError
labels = load_labels(ROOT / "images/labels.csv")
files = [reference_png] + sorted(p for p in (ROOT / "images/own").rglob("*")
                                 if p.is_file() and p.suffix.lower() in EXTENSIONS)
results = []
for path in files:
    try:
        _ = read_rgb(path)
    except (UnidentifiedImageError, OSError) as exc:
        results.append({"file": str(path.relative_to(ROOT)), "status": "image_error", "error": str(exc)})
        continue
    result, _, _ = infer_file(classifier, path)
    expected = expected_label(path, labels)
    result.update(status="ok", expected_id="" if expected is None else expected,
                  correct="" if expected is None else int(result["prediction_id"] == expected))
    results.append(result)
    print("{} -> {} | asteptat: {}".format(result["file"], result["prediction"],
          "fara eticheta" if expected is None else CLASS_NAMES[expected]))
save_csv(ROOT / "results/classification_results.csv", results)
valid = [r for r in results if r["status"] == "ok"]
labelled = [r for r in valid if r["expected_id"] != ""]
print("Procesate:", len(valid), "| erori:", len(results)-len(valid), "| etichetate:", len(labelled))
if labelled:
    correct = sum(r["correct"] for r in labelled)
    print("Acuratete pe acest lot: {}/{} = {:.2f}%".format(correct, len(labelled), 100*correct/len(labelled)))
else:
    print("Fara etichete: nu calculam acuratetea.")

## 7. Benchmark fara afisare

O incalzire, apoi 20 de repetari pe aceeasi imagine. Fisierul poate fi deja in cache-ul Linux. Incarcarea overlay-ului, afisarea si scrierea CSV sunt excluse.

`api_ms` include preprocesarea, fisierul temporar, transferurile si inferenta prin API-ul original. `library_reported_ms` este timpul intern raportat de biblioteca: nu il numim automat timp pur al circuitului. `file_to_prediction_ms` include citirea, API-ul si alegerea clasei.

Aceste durate se suprapun: nu le adunati. Nu sunt FPS-ul unei camere.

In [ ]:
N_REPEATS = 20
if not isinstance(N_REPEATS, int) or not 1 <= N_REPEATS <= 1000:
    raise ValueError("N_REPEATS trebuie sa fie intre 1 si 1000.")
_ = infer_file(classifier, IMAGE_PATH)
samples = [infer_file(classifier, IMAGE_PATH)[0] for _ in range(N_REPEATS)]
save_csv(ROOT / "results/benchmark_samples.csv", samples)
summary = []
for metric in ("decode_ms", "api_ms", "library_reported_ms", "file_to_prediction_ms"):
    values = np.array([sample[metric] for sample in samples], dtype=float)
    row = {"metric": metric, "n": N_REPEATS, "mean_ms": float(np.mean(values)),
           "median_ms": float(np.median(values)), "p95_ms": float(np.percentile(values, 95))}
    summary.append(row)
    print("{:25s} medie={:.3f} ms | mediana={:.3f} ms | p95={:.3f} ms".format(
        metric, row["mean_ms"], row["median_ms"], row["p95_ms"]))
save_csv(ROOT / "results/benchmark_summary.csv", summary)
print("Predictii stabile:", len({row["prediction_id"] for row in samples}) == 1)

## 8. Sarcini si raport

1. Adaugati minimum sase imagini proprii din cel putin trei clase suportate. Raportati separat rezultatele lor si referinta.
2. Comparati o imagine clasificata gresit cu un decupaj manual al aceluiasi obiect. Vizualizati intrarea efectiva 32 x 32. Decupajul manual nu este detectie automata.
3. Explicati diferentele dintre timpii masurati. Testati si un obiect nesuportat: de ce apare tot o eticheta?

Predati `classification_results.csv`, `benchmark_summary.csv`, `environment.json` si doua capturi explicate. Nu selectati numai exemplele reusite.

La final opriti kernelul. Pentru alt laborator incarcati overlay-ul lui in mod obisnuit. Fisierele Lab1-Lab6 raman neschimbate.

## Referinte si validare

[Exemplul CNV-QNN](https://github.com/onigas/BNN-PYNQ/blob/d39c1c75403db571f54dd0f0cf27ae5104ac968e/notebooks/CNV-QNN_Cifar10.ipynb), [testele originale](https://github.com/onigas/BNN-PYNQ/blob/d39c1c75403db571f54dd0f0cf27ae5104ac968e/tests/test_qnn.py), [API-ul original](https://github.com/onigas/BNN-PYNQ/blob/d39c1c75403db571f54dd0f0cf27ae5104ac968e/bnn/bnn.py).

Modelul provine dintr-un proiect testat anterior, dar acest notebook nu a fost executat pe PYNQ-Z2 in mediul de pregatire. Iesirile sunt sterse: PASS, predictiile si timpii se obtin numai prin rularea efectiva. Vezi `VALIDATION.md`.